In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "leeuwen2013total")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "VanLeeuwen_2013_perceptions_tab_not_original.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['studyid']="leeuwen2013total"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns

In [3]:
tai = df[['studyid', 'ape', 'day', 'tai_white', 'tai_brown',
    'unnamed: 0', 'unnamed: 1']]
tai = tai.assign(ape2='tai')

pia = df[['studyid', 'ape', 'day', 'pia_white', 'pia_brown',
    'unnamed: 0', 'unnamed: 1']]
pia = pia.assign(ape2='pia')

lome = df[['studyid', 'ape', 'day', 'lome_white', 'lome_brown',
    'unnamed: 0', 'unnamed: 1']]
lome = lome.assign(ape2='lome')

fraukje = df[['studyid', 'ape', 'day', 'fraukje_white', 'fraukje_brown',
    'unnamed: 0', 'unnamed: 1']]
fraukje = fraukje.assign(ape2='fraukje')

kara = df[['studyid', 'ape', 'day', 'kara_white', 'kara_brown', 
    'unnamed: 0', 'unnamed: 1']]
kara = kara.assign(ape2='kara')

kofi = df[['studyid', 'ape', 'day', 'kofi_white', 'kofi_brown',
    'unnamed: 0', 'unnamed: 1']]
kofi = kofi.assign(ape2='kofi')

lobo = df[['studyid', 'ape', 'day', 'lobo_white', 'lobo_brown',
    'unnamed: 0', 'unnamed: 1']]
lobo = lobo.assign(ape2='lobo')

swela = df[['studyid', 'ape', 'day', 'swela_white', 'swela_brown',
    'unnamed: 0', 'unnamed: 1']]
swela = swela.assign(ape2='swela')

ulla = df[['studyid', 'ape', 'day', 'ulla_white', 'ulla_brown',
    'unnamed: 0', 'unnamed: 1']]
ulla = ulla.assign(ape2='ulla')

dorien = df[['studyid', 'ape', 'day', 'dorien_white', 'dorien_brown',
    'unnamed: 0', 'unnamed: 1']]
dorien = dorien.assign(ape2='dorien')

In [4]:
data_frames=[tai, pia, lome, fraukje, kara, kofi, lobo, swela, ulla, dorien]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x['ape'] = x['ape'].str.rstrip()
    x['ape2'] = x['ape2'].str.rstrip()
    x.columns = x.columns.str.replace(r'.*_', '', regex=True)
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

In [5]:
fulldf.rename(columns={"studyid":"study_id"}, inplace=True)


In [6]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
fulldf['ape2'] = fulldf['ape2'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)
    fulldf['ape2'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')



In [7]:
code_list = fulldf[["ape", "ape2"]]
temp = []
temp1 = []
for index, row in code_list.iterrows():
    if row["ape"] == row["ape2"]:
        temp.append(np.nan)
        temp1.append("")
    elif row["ape"] != row["ape2"]: 
        temp.append(row["ape2"])
        temp1.append(str(row["ape"])+ "_" + str(row["ape2"]))
    else:
        temp.append("")
        temp1.append("")
fulldf = fulldf.assign(temp_col=temp)
fulldf = fulldf.assign(temp_col1=temp1)
fulldf=fulldf.rename(columns={'temp_col': 'ape_2', 
    'temp_col1': 'dyad'})

In [8]:
comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
fulldf= fulldf.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

In [9]:
role=[]
role_2=[]
for index, row in fulldf.iterrows():
    if not pd.isna(row['ape']) and not pd.isna(row['ape_2']):
        role.append("observed")
    # elif not pd.isna(row['ape']) and pd.isna(row['ape_2']):
    #     role.append("exchanger")
    else:
        role.append("exchanger")
fulldf = fulldf.assign(role=role)
for index, row in fulldf.iterrows(): 
    if not pd.isna(row['ape_2']):
        role_2.append("observer")
    else:
        role_2.append("")
fulldf = fulldf.assign(role_2=role_2)

In [10]:
fulldf.dropna(subset=['ape'], inplace=True)
# fulldf.columns
fulldf.rename(columns={"ape": "participant", "ape_2":"participant_2"}, inplace=True)

In [11]:
complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
fulldf= fulldf.merge(subject_list,left_on='participant', right_on='name', how='left')
fulldf.rename(columns={"age": "age_in_years"}, inplace=True)

complete_path_age = os.path.join(original_data_pathway, "subject_list_2.csv")
subject_list = pd.read_csv(complete_path_age)   
fulldf= fulldf.merge(subject_list,left_on='participant_2', right_on='name_2', how='left')
fulldf.rename(columns={"age_2": "age_in_years_2"}, inplace=True)

In [12]:
leeuwen2013total_standardized=fulldf[['study_id', 'participant','age_in_years','sex', 'role',
                                      'participant_2',  'age_in_years_2',
    'sex_2', 'role_2','dyad', 'species',
    'day' , 'white', 'brown']]
comp_out_path_stand = os.path.join(out_pathway, 'leeuwen2013total_exp1_standardized.csv')
leeuwen2013total_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =leeuwen2013total_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
leeuwen2013total_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'leeuwen2013total_exp1_glossary.csv')
leeuwen2013total_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
